In [6]:
import os
import pandas as pd
import xgboost as xgb
import numpy as np

from pathlib import Path
import sys
sys.path.append("/home/alecacciatore/rolex/deep-nir/src/deep_nir/")

from VRAI.main_train_xgb import main
from VRAI.utils.models import train_xgb_regr, train_xgb_class
from VRAI.utils.data import y_columns, get_x_y_labels, col_names_switch

# Train XGBoost regressor

## Get data

In [7]:
in_path = "/home/alecacciatore/rolex/deep-nir/data/raw/Grain"
beams_step = 1
sheet_names = ["DATASET"]

X_train_all, X_val_all, y_train_all, y_val_all = {}, {}, {}, {}
for dir in os.listdir(in_path):
    dataset_name = dir.split('_')[1][:-len('Grain')]
    for file in os.listdir(os.path.join(in_path, dir)):
        if file.endswith("DATASET.xlsx"):
            print(f"\nProcessing dataset: {dataset_name}, file: {file}")
            X_train_all[dataset_name], X_val_all[dataset_name], y_train_all[dataset_name], y_val_all[dataset_name] = main(dir, sheet_names, os.path.join(in_path, dir, file), beams_step)
   


Processing dataset: Soybean, file: 59-SG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Barley, file: 25-OR_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Rapeseed, file: 143-RG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Corn, file: 08-MG_Rev489_DATASET.xlsx
	Target columns: ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']

Processing dataset: Wheat, file: 09-FG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']


## Train model

In [8]:
output_csv = None
results = {}

for y_col in y_train_all[list(y_train_all.keys())[0]].keys():
    print(f"\nTraining combined models for target: {y_col}")
    X_train_combined = pd.concat([X_train_all[dataset_name][y_col] for dataset_name in X_train_all])
    y_train_combined = pd.concat([y_train_all[dataset_name][y_col] for dataset_name in y_train_all])
    X_val_combined = pd.concat([X_val_all[dataset_name][y_col] for dataset_name in X_val_all])
    y_val_combined = pd.concat([y_val_all[dataset_name][y_col] for dataset_name in y_val_all])

    # Add dataset classification labels
    train_dirs = pd.Series([dataset_name for dataset_name in X_train_all for _ in range(len(X_train_all[dataset_name][y_col]))], name="dataset_label")
    val_dirs = pd.Series([dataset_name for dataset_name in X_val_all for _ in range(len(X_val_all[dataset_name][y_col]))], name="dataset_label")
    
    # Train regression and classification models
    print(f"\nTraining combined XGBoost model for target: {y_col}")
    results[y_col] = train_xgb_regr(X_train_combined, X_val_combined, y_train_combined, y_val_combined, y_col, output_csv, beams_step)


Training combined models for target: DM

Training combined XGBoost model for target: DM


Numero bande selezionate da XGB: 36
Indici bande selezionate: [ 4  5  9 10 11 13 14 15 21 22 23 24 28 29 30 31 32 37 38 39 40 41 42 43
 44 45 46 47 48 58 59 60 61 62 63 70]
Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 400, 'model__subsample': 0.7}
MAE: 0.44417739766517855 R2: 0.939417663653256
Validation RMSE: 0.2203180826742284
Training RMSE: 0.24251354483530413

Training combined models for target: Starch

Training combined XGBoost model for target: Starch
Numero bande selezionate da XGB: 36
Indici bande selezionate: [ 5  7 17 18 19 20 21 22 23 25 34 35 36 37 38 39 41 43 44 45 46 48 49 51
 52 53 54 58 60 62 63 64 65 67 68 69]
Best params: {'model__colsample_bytree': 0.7, 'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 0.9}
MAE: 1.0745758303215964 R2: 0.9528626718710341
Validation RMSE: 0.42406790976297826
Training RMSE: 0.5140556834810847

Training combine

In [12]:
results['DM'].keys()

dict_keys(['model', 'selected_indices', 'x_val', 'y_val', 'predictions', 'metrics', 'train_metrics'])

In [16]:
from joblib import dump

out_path = "/home/alecacciatore/rolex/deep-nir/src/deep_nir/VRAI/outputs/pretrained_models"

# Salva il modello addestrato
for y_col, result in results.items():
    model_out_path = os.path.join(out_path, f"{y_col}_xgb_regr.joblib")
    dump(result["model"], model_out_path)